In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [43]:
class ScaledRFPDataset(Dataset):
    def __init__(self, pt_file_path, mu, sigma):
        data = torch.load(pt_file_path, weights_only=False)
        self.embeddings = data['embeddings']
        raw_log_prices = data['log_prices']
        self.targets = (raw_log_prices - mu) / sigma
        
    def __len__(self): return len(self.embeddings)
    def __getitem__(self, idx):
        return self.embeddings[idx], torch.tensor(self.targets[idx], dtype=torch.float32)
    
def tube_loss(y, mu1, mu2, t=0.95, r=0.5, delta=0.0):
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)
    
    loss = torch.zeros_like(y)
    
    loss[y > upper] = t * (y[y > upper] - upper[y > upper])
    loss[y < lower] = t * (lower[y < lower] - y[y < lower])

    mid = r * upper + (1 - r) * lower
    mask = (y >= lower) & (y <= upper)
    loss[mask & (y >= mid)] = (1 - t) * (upper[mask & (y >= mid)] - y[mask & (y >= mid)])
    loss[mask & (y < mid)]  = (1 - t) * (y[mask & (y < mid)] - lower[mask & (y < mid)])
    
    base_loss = loss.mean()
    
    width_penalty = torch.abs(upper - lower).mean()
    
    return base_loss + (delta * width_penalty)


In [3]:
class CrossAttentionCategoryHead(nn.Module):
    def __init__(self, embed_dim=768, num_heads=4):
        # Note: 1536-D vector is actually two 768-D vectors (image + text) concatenated.
        super().__init__()
        
        # The Cross-Attention Layer: Text attends to Image
        self.cross_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        
        self.fc_head = nn.Sequential(
            nn.Linear(embed_dim * 2, 512), 
            nn.LayerNorm(512), 
            nn.GELU(), 
            nn.Dropout(0.3),
            nn.Linear(512, 256), 
            nn.GELU(), 
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = x.float()
        
        img_emb = x[:, :768].unsqueeze(1)  
        txt_emb = x[:, 768:].unsqueeze(1) 

        # Cross Attention: Q = Image, K = Text, V = Text
        attn_out, _ = self.cross_attn(query=img_emb, key=txt_emb, value=txt_emb)
        
        attn_out = attn_out.squeeze(1)
        img_out = img_emb.squeeze(1)
        
        fused_features = torch.cat([img_out, attn_out], dim=1) # Shape: (Batch, 1536)
        
        return self.fc_head(fused_features)

In [4]:
def calculate_interval_score(y_true, lower, upper, alpha=0.05):
    """
    Calculates the Interval Score (IS) for 95% confidence intervals (alpha=0.05).
    y_true, lower, and upper should be PyTorch tensors.
    """

    width = upper - lower
    
    #Penalty if the true price is lower than the lower bound
    penalty_lower = (2.0 / alpha) * (lower - y_true)
    penalty_lower = torch.where(y_true < lower, penalty_lower, torch.zeros_like(width))
    
    #Penalty if the true price is higher than the upper bound
    penalty_upper = (2.0 / alpha) * (y_true - upper)
    penalty_upper = torch.where(y_true > upper, penalty_upper, torch.zeros_like(width))
    
    #final score is the width plus penalties for missing the target
    interval_score = width + penalty_lower + penalty_upper
    
    return interval_score.mean().item()

In [14]:
def train_cross_attention_model(split_dir="split_embeddings", epochs=15):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}\n")
    
    category_t_map = {
        "BIKES": 0.94, "BOOKS": 0.97, "CARS": 0.96, "CYCLE": 0.97,
        "FLAT": 0.96, "FRIDGES": 0.97, "GAMES": 0.96, "GAMESENTERTAINMENT": 0.96,
        "LAPTOP": 0.97, "PHONES": 0.98, "PRINTER": 0.95,
        "TV": 0.97, "WASHINGMACHINE": 0.94
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        t_val = category_t_map.get(cat_name, 0.95)
        alpha = 1.0 - t_val  # Required for Interval Score calculation
        
        print(f"\nCROSS-ATTENTION MODEL: {cat_name}\n{'-'*50}")
        
        # Load Data
        raw_train_data = torch.load(train_path, weights_only=False)
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        train_dataset = ScaledRFPDataset(train_path, cat_mu, cat_sigma)
        test_dataset = ScaledRFPDataset(test_path, cat_mu, cat_sigma)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
        
        # Initialize Cross-Attention Model
        model = CrossAttentionCategoryHead().to(device)
        optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        # Train
        model.train()
        for epoch in range(epochs):
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                
                optimizer.zero_grad()
                out = model(x)
                loss = tube_loss(y, out[:,0], out[:,1], t=t_val, r=0.5)
                loss.backward()
                optimizer.step()

        model.eval()
        lowers, uppers, ys = [], [], []
        
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_device = x_batch.to(device)
                
                out = model(x_device)
                l = torch.min(out[:,0], out[:,1])
                u = torch.max(out[:,0], out[:,1])
                
                # Inverse Transform to real Rupees
                l_real = torch.exp(l * cat_sigma + cat_mu)
                u_real = torch.exp(u * cat_sigma + cat_mu)
                y_true = torch.exp(y_batch.to(device) * cat_sigma + cat_mu)
                
                lowers.append(l_real)
                uppers.append(u_real)
                ys.append(y_true)
                
        l_preds = torch.cat(lowers)
        u_preds = torch.cat(uppers)
        y_trues = torch.cat(ys)
        
        # Metrics Calculations
        picp = ((y_trues >= l_preds) & (y_trues <= u_preds)).float().mean().item()
        mpiw = (u_preds - l_preds).mean().item()
        mid_preds = (l_preds + u_preds) / 2.0
        rmse = torch.sqrt(torch.mean((y_trues - mid_preds) ** 2)).item()
        
        interval_score = calculate_interval_score(y_trues, l_preds, u_preds, alpha=alpha)
        
        print(f"PICP (Coverage):   {picp:.4f}")
        print(f"MPIW (Width):      ₹{mpiw:,.2f}")
        print(f"RMSE (Midpoint):   ₹{rmse:,.2f}")
        print(f"Interval Score:    {interval_score:,.2f}")

In [15]:
torch.manual_seed(54)
train_cross_attention_model(epochs=5)

Using device: cpu


CROSS-ATTENTION MODEL: BIKES
--------------------------------------------------
PICP (Coverage):   0.9773
MPIW (Width):      ₹408,490.72
RMSE (Midpoint):   ₹135,383.16
Interval Score:    482,322.59

CROSS-ATTENTION MODEL: BOOKS
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹20,218.55
RMSE (Midpoint):   ₹8,618.37
Interval Score:    20,218.55

CROSS-ATTENTION MODEL: CARS
--------------------------------------------------
PICP (Coverage):   0.9677
MPIW (Width):      ₹11,269,866.00
RMSE (Midpoint):   ₹5,136,319.00
Interval Score:    11,334,981.00

CROSS-ATTENTION MODEL: CYCLE
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹1,108,192.12
RMSE (Midpoint):   ₹548,697.81
Interval Score:    1,108,192.12

CROSS-ATTENTION MODEL: FLAT
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹22,307,436.00
RMSE (Midpoint):   ₹5,782,008.50
Inter

In [71]:
torch.cuda.empty_cache()

In [16]:
class GatedMultimodalHead(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        
        self.img_transform = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.txt_transform = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # It takes both original embeddings and outputs a value between 0 and 1
        self.gate_layer = nn.Sequential(
            nn.Linear(embed_dim * 2, 512),
            nn.Sigmoid() # Squashes output to (0, 1)
        )
        
        self.fc_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = x.float()
        
        # Split into Image and Text
        img_emb = x[:, :768]
        txt_emb = x[:, 768:]
        
        # Transform them individually
        img_feat = self.img_transform(img_emb)
        txt_feat = self.txt_transform(txt_emb)
        
        # Calculate the Gate
        # Gate = 1 means trust Image perfectly. Gate = 0 means trust Text perfectly.
        gate = self.gate_layer(x)
        
        # Fuse them using the Gate (This is the magic equation from the IJCAI review)
        fused_features = gate * img_feat + (1 - gate) * txt_feat
        
        # Predict
        return self.fc_head(fused_features)

In [41]:
def train_gated_multimodal_head_model(split_dir="split_embeddings", epochs=15):
    # Change to "cpu" or "cuda:1" if GPU 0 is blocked
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}\n")
    
    category_t_map = {
        "BIKES": 0.95, "BOOKS": 0.96, "CARS": 0.95, "CYCLE": 0.95,
        "FLAT": 0.96, "FRIDGES": 0.98, "GAMES": 0.96, "GAMESENTERTAINMENT": 0.97,
        "LAPTOP": 0.97, "PHONES": 0.98, "PRINTER": 0.95,
        "TV": 0.97, "WASHINGMACHINE": 0.95
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        t_val = category_t_map.get(cat_name,0.95)
        alpha = 1.0 - t_val
        
        print(f"\nFEATURE-WISE GATING MODEL: {cat_name}\n{'-'*50}")
        
        # Load Data
        raw_train_data = torch.load(train_path, weights_only=False)
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        train_dataset = ScaledRFPDataset(train_path, cat_mu, cat_sigma)
        test_dataset = ScaledRFPDataset(test_path, cat_mu, cat_sigma)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
        
        # Initialize Cross-Attention Model
        model = GatedMultimodalHead().to(device)
        optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        # Train
        model.train()
        for epoch in range(epochs):
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                
                optimizer.zero_grad()
                out = model(x)
                loss = tube_loss(y, out[:,0], out[:,1], t=t_val, r=0.5)
                loss.backward()
                optimizer.step()

        # Evaluate
        model.eval()
        lowers, uppers, ys = [], [], []
        
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_device = x_batch.to(device)
                
                out = model(x_device)
                l = torch.min(out[:,0], out[:,1])
                u = torch.max(out[:,0], out[:,1])
                
                # Inverse Transform to real Rupees
                l_real = torch.exp(l * cat_sigma + cat_mu)
                u_real = torch.exp(u * cat_sigma + cat_mu)
                y_true = torch.exp(y_batch.to(device) * cat_sigma + cat_mu)
                
                lowers.append(l_real)
                uppers.append(u_real)
                ys.append(y_true)
                
        l_preds = torch.cat(lowers)
        u_preds = torch.cat(uppers)
        y_trues = torch.cat(ys)
        
        # Metrics Calculations
        picp = ((y_trues >= l_preds) & (y_trues <= u_preds)).float().mean().item()
        mpiw = (u_preds - l_preds).mean().item()
        mid_preds = (l_preds + u_preds) / 2.0
        rmse = torch.sqrt(torch.mean((y_trues - mid_preds) ** 2)).item()
        
        # IJCAI Requirement: Interval Score
        interval_score = calculate_interval_score(y_trues, l_preds, u_preds, alpha=alpha)
        
        print(f"PICP (Coverage):   {picp:.4f}")
        print(f"MPIW (Width):      ₹{mpiw:,.2f}")
        print(f"RMSE (Midpoint):   ₹{rmse:,.2f}")
        print(f"Interval Score:    {interval_score:,.2f}")

In [42]:
torch.cuda.empty_cache()
torch.manual_seed(54)
train_gated_multimodal_head_model(epochs=5)

Using device: cpu


FEATURE-WISE GATING MODEL: BIKES
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹481,387.81
RMSE (Midpoint):   ₹160,338.72
Interval Score:    481,387.81

FEATURE-WISE GATING MODEL: BOOKS
--------------------------------------------------
PICP (Coverage):   0.9565
MPIW (Width):      ₹9,419.18
RMSE (Midpoint):   ₹3,928.16
Interval Score:    16,452.37

FEATURE-WISE GATING MODEL: CARS
--------------------------------------------------
PICP (Coverage):   0.9677
MPIW (Width):      ₹2,784,558.25
RMSE (Midpoint):   ₹1,020,660.75
Interval Score:    3,099,045.75

FEATURE-WISE GATING MODEL: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9655
MPIW (Width):      ₹20,710.76
RMSE (Midpoint):   ₹8,777.79
Interval Score:    36,952.60

FEATURE-WISE GATING MODEL: FLAT
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹49,556,996.00
RMSE (Midpoint):   ₹19,526,28